# Clustering de profils joueurs
Objectif : regrouper les sessions de jeu selon les **comportements d'entrée** (boutons, joystick, réactivité, score) pour identifier des profils joueurs.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, DBSCAN
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score, silhouette_samples
from sklearn.manifold import TSNE
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (10, 6)

## 1. Chargement et exploration des données

In [ ]:
df = pd.read_csv('sessions_rows.csv')
print(f'Shape : {df.shape}')
df.head()

In [ ]:
df.info()

In [ ]:
df.describe().round(3)

In [ ]:
print('Jeux :', df['game_id'].value_counts().to_dict())
print('Sources :', df['source'].value_counts().to_dict())
print('Joueurs :', df['player_name'].value_counts().to_dict())

In [ ]:
# Distributions des variables clés
key_vars = ['btn_press_rate', 'btn_hold_avg_ms', 'reaction_time_avg_ms',
            'input_regularity', 'score', 'duration_sec']

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for ax, col in zip(axes.flat, key_vars):
    df[col].hist(ax=ax, bins=15, edgecolor='white', color='steelblue')
    ax.set_title(col)
    ax.set_xlabel('')
plt.suptitle('Distribution des variables comportementales', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# Corrélation
numeric_cols = df.select_dtypes(include=np.number).columns
plt.figure(figsize=(14, 10))
sns.heatmap(df[numeric_cols].corr(), annot=True, fmt='.2f', cmap='coolwarm',
            center=0, linewidths=0.5)
plt.title('Matrice de corrélation')
plt.tight_layout()
plt.show()

## 2. Préparation des features pour le clustering

On sélectionne les features comportementales qui décrivent le **style de jeu** :
- **Activité boutons** : fréquence, variété, durée appui
- **Joystick** : amplitude et précision des mouvements
- **Performance** : temps de réaction, régularité, score
- **Agressivité** : brutality triggers

In [ ]:
# --- Gestion des variables corrélées ---
# lx_std & ly_std (r=0.94) → on combine en magnitude joystick gauche
# rx_std & ry_std (r=0.93) → idem pour joystick droit
df['lx_amplitude'] = np.sqrt(df['lx_std']**2 + df['ly_std']**2)
df['rx_amplitude'] = np.sqrt(df['rx_std']**2 + df['ry_std']**2)

FEATURES = [
    'btn_press_rate',       # Fréquence d'appui boutons
    'btn_variety',          # Diversité des boutons utilisés
    'btn_hold_avg_ms',      # Durée moyenne d'appui (style)
    'lx_amplitude',         # Amplitude joystick gauche (lx_std + ly_std combinés)
    'rx_amplitude',         # Amplitude joystick droit  (rx_std + ry_std combinés)
    'lx_direction_changes', # Dynamisme joystick gauche
    'lt_brutality',         # Agressivité gâchette gauche
    'rt_brutality',         # Agressivité gâchette droite
    'reaction_time_avg_ms', # Réactivité
    'input_regularity',     # Régularité des inputs
    'score',                # Performance
]

X = df[FEATURES].copy()
print('Valeurs manquantes par feature :')
print(X.isnull().sum())
X = X.fillna(0)

In [ ]:
# Normalisation
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_scaled_df = pd.DataFrame(X_scaled, columns=FEATURES)
print('Features normalisées - moyennes :', X_scaled_df.mean().round(3).to_dict())

## 3. Choix du nombre de clusters

In [ ]:
inertias = []
silhouettes = []
K_range = range(2, 10)

for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_scaled)
    inertias.append(km.inertia_)
    silhouettes.append(silhouette_score(X_scaled, labels))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Méthode du coude
ax1.plot(list(K_range), inertias, 'bo-', linewidth=2, markersize=8)
ax1.set_xlabel('Nombre de clusters (k)')
ax1.set_ylabel('Inertie (WCSS)')
ax1.set_title('Méthode du coude')
ax1.grid(True)

# Score silhouette
ax2.plot(list(K_range), silhouettes, 'rs-', linewidth=2, markersize=8)
ax2.set_xlabel('Nombre de clusters (k)')
ax2.set_ylabel('Score silhouette')
ax2.set_title('Score silhouette (plus = mieux)')
ax2.grid(True)

best_k = list(K_range)[np.argmax(silhouettes)]
ax2.axvline(x=best_k, color='green', linestyle='--', label=f'Meilleur k={best_k}')
ax2.legend()

plt.suptitle('Sélection du nombre optimal de clusters', fontsize=14)
plt.tight_layout()
plt.show()

print(f'\nMeilleur k selon silhouette : {best_k} (score = {max(silhouettes):.3f})')

## 4. K-Means avec le k optimal

In [ ]:
K_OPTIMAL = best_k  # Modifier manuellement si besoin

kmeans = KMeans(n_clusters=K_OPTIMAL, random_state=42, n_init=10)
df['cluster'] = kmeans.fit_predict(X_scaled)

print(f'Clusters formés ({K_OPTIMAL} groupes) :')
print(df['cluster'].value_counts().sort_index())
print(f'\nScore silhouette final : {silhouette_score(X_scaled, df["cluster"]):.3f}')

In [ ]:
# Visualisation silhouette par cluster
fig, ax = plt.subplots(figsize=(10, 6))
sample_sil = silhouette_samples(X_scaled, df['cluster'])
y_lower = 10
colors = cm.nipy_spectral(np.linspace(0, 1, K_OPTIMAL))

for i in range(K_OPTIMAL):
    sil_i = sorted(sample_sil[df['cluster'] == i])
    size_i = len(sil_i)
    y_upper = y_lower + size_i
    ax.fill_betweenx(np.arange(y_lower, y_upper), 0, sil_i, facecolor=colors[i], alpha=0.7)
    ax.text(-0.05, y_lower + 0.5 * size_i, f'C{i}')
    y_lower = y_upper + 10

ax.axvline(x=silhouette_score(X_scaled, df['cluster']), color='red', linestyle='--', label='Moyenne')
ax.set_xlabel('Coefficient silhouette')
ax.set_title('Silhouette par cluster')
ax.legend()
plt.tight_layout()
plt.show()

## 5. Visualisation PCA 2D

In [ ]:
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)

print(f'Variance expliquée par PCA : {pca.explained_variance_ratio_.sum():.1%}')
print(f'  PC1 : {pca.explained_variance_ratio_[0]:.1%}')
print(f'  PC2 : {pca.explained_variance_ratio_[1]:.1%}')

df_plot = pd.DataFrame({
    'PC1': X_pca[:, 0],
    'PC2': X_pca[:, 1],
    'cluster': df['cluster'].astype(str),
    'game_id': df['game_id'],
    'source': df['source'].fillna('unknown'),
    'player_name': df['player_name'],
    'score': df['score'],
})

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# Par cluster
sns.scatterplot(data=df_plot, x='PC1', y='PC2', hue='cluster',
                palette='tab10', s=100, ax=axes[0])
axes[0].set_title('Clusters K-Means (PCA 2D)')

# Par jeu
sns.scatterplot(data=df_plot, x='PC1', y='PC2', hue='game_id',
                palette='Set2', s=100, ax=axes[1])
axes[1].set_title('Par type de jeu')

# Par source (keyboard / controller)
sns.scatterplot(data=df_plot, x='PC1', y='PC2', hue='source',
                palette='Set1', s=100, ax=axes[2])
axes[2].set_title('Par source (keyboard / controller)')

plt.tight_layout()
plt.show()

## 6. Profil de chaque cluster

In [ ]:
# Moyennes par cluster
cluster_profile = df.groupby('cluster')[FEATURES].mean().round(3)
cluster_profile

In [ ]:
# Radar chart des profils (features normalisées)
cluster_profile_norm = pd.DataFrame(
    scaler.transform(cluster_profile),
    columns=FEATURES,
    index=cluster_profile.index
)

# Heatmap des profils normalisés
plt.figure(figsize=(14, 5))
sns.heatmap(cluster_profile_norm.T, annot=True, fmt='.2f', cmap='RdYlGn',
            center=0, linewidths=0.5,
            xticklabels=[f'Cluster {i}' for i in cluster_profile.index])
plt.title('Profil des clusters (valeurs normalisées — vert = élevé, rouge = faible)')
plt.tight_layout()
plt.show()

In [ ]:
# Répartition par jeu et source dans chaque cluster
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

pd.crosstab(df['cluster'], df['game_id'], normalize='index').plot(
    kind='bar', ax=axes[0], stacked=True, colormap='Set2')
axes[0].set_title('Répartition des jeux par cluster')
axes[0].set_xlabel('Cluster')
axes[0].set_ylabel('Proportion')
axes[0].tick_params(axis='x', rotation=0)

pd.crosstab(df['cluster'], df['source'].fillna('unknown'), normalize='index').plot(
    kind='bar', ax=axes[1], stacked=True, colormap='Set1')
axes[1].set_title('Répartition des sources par cluster')
axes[1].set_xlabel('Cluster')
axes[1].tick_params(axis='x', rotation=0)

df.groupby('cluster')['score'].mean().plot(
    kind='bar', ax=axes[2], color='steelblue', edgecolor='white')
axes[2].set_title('Score moyen par cluster')
axes[2].set_xlabel('Cluster')
axes[2].tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.show()

In [ ]:
# Boxplots des features clés par cluster
KEY_PLOT = ['btn_press_rate', 'reaction_time_avg_ms', 'score', 'input_regularity']

fig, axes = plt.subplots(1, 4, figsize=(18, 5))
for ax, col in zip(axes, KEY_PLOT):
    df.boxplot(column=col, by='cluster', ax=ax)
    ax.set_title(col)
    ax.set_xlabel('Cluster')

plt.suptitle('')
plt.tight_layout()
plt.show()

## 6b. Visualisations avancées des clusters

In [ ]:
# --- RADAR CHART (spider plot) par cluster ---
from matplotlib.patches import FancyArrowPatch
import matplotlib.patches as mpatches

cluster_means = df.groupby('cluster')[FEATURES].mean()
# Normaliser entre 0 et 1 pour le radar
cluster_means_norm = (cluster_means - cluster_means.min()) / (cluster_means.max() - cluster_means.min() + 1e-9)

labels = FEATURES
N = len(labels)
angles = np.linspace(0, 2 * np.pi, N, endpoint=False).tolist()
angles += angles[:1]  # fermer le polygone

palette = plt.cm.tab10(np.linspace(0, 1, K_OPTIMAL))

fig, axes = plt.subplots(1, K_OPTIMAL, figsize=(5 * K_OPTIMAL, 5),
                         subplot_kw=dict(polar=True))
if K_OPTIMAL == 1:
    axes = [axes]

for i, ax in enumerate(axes):
    values = cluster_means_norm.iloc[i].tolist()
    values += values[:1]
    ax.plot(angles, values, color=palette[i], linewidth=2)
    ax.fill(angles, values, color=palette[i], alpha=0.3)
    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(labels, size=8)
    ax.set_yticks([0.25, 0.5, 0.75, 1.0])
    ax.set_yticklabels(['25%', '50%', '75%', '100%'], size=6, color='grey')
    ax.set_title(f'Cluster {i}\n({(df["cluster"]==i).sum()} sessions)',
                 size=12, pad=15, color=palette[i])

plt.suptitle('Radar chart — profil de chaque cluster', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# --- t-SNE (alternative non-linéaire à la PCA) ---
tsne = TSNE(n_components=2, random_state=42, perplexity=min(30, len(X_scaled)-1))
X_tsne = tsne.fit_transform(X_scaled)

df_tsne = pd.DataFrame({
    'tSNE1': X_tsne[:, 0],
    'tSNE2': X_tsne[:, 1],
    'cluster': df['cluster'].astype(str),
    'game_id': df['game_id'],
    'source': df['source'].fillna('unknown'),
    'player_name': df['player_name'],
})

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# t-SNE coloré par cluster
sns.scatterplot(data=df_tsne, x='tSNE1', y='tSNE2', hue='cluster',
                palette='tab10', s=120, ax=axes[0])
# Annoter les noms des joueurs
for _, row in df_tsne.iterrows():
    axes[0].annotate(row['player_name'], (row['tSNE1'], row['tSNE2']),
                     fontsize=6, alpha=0.6, ha='center', va='bottom')
axes[0].set_title('t-SNE — coloré par cluster')

# t-SNE coloré par jeu
sns.scatterplot(data=df_tsne, x='tSNE1', y='tSNE2', hue='game_id',
                palette='Set2', s=120, style='source', ax=axes[1])
axes[1].set_title('t-SNE — coloré par jeu (marqueur = source)')

plt.suptitle('t-SNE : représentation non-linéaire des sessions', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# --- PARALLEL COORDINATES --- toutes les features simultanément par cluster
from pandas.plotting import parallel_coordinates

df_para = X_scaled_df.copy()
df_para['cluster'] = 'Cluster ' + df['cluster'].astype(str)

palette_para = {f'Cluster {i}': plt.cm.tab10(i / K_OPTIMAL) for i in range(K_OPTIMAL)}

plt.figure(figsize=(16, 6))
parallel_coordinates(df_para, class_column='cluster',
                     color=[palette_para[c] for c in sorted(df_para['cluster'].unique())],
                     alpha=0.4, linewidth=1.5)
plt.xticks(rotation=30, ha='right')
plt.title('Parallel coordinates — toutes les features par cluster\n(chaque ligne = une session)', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# --- PAIRPLOT — scatter matrix des features clés, coloré par cluster ---
PAIR_FEATURES = ['btn_press_rate', 'score', 'reaction_time_avg_ms',
                 'input_regularity', 'lx_amplitude', 'lt_brutality']

df_pair = df[PAIR_FEATURES].copy()
df_pair['cluster'] = 'C' + df['cluster'].astype(str)

g = sns.pairplot(df_pair, hue='cluster', palette='tab10',
                 diag_kind='kde', plot_kws={'alpha': 0.6, 's': 60},
                 corner=True)
g.figure.suptitle('Pairplot des features clés — coloré par cluster', y=1.02, fontsize=13)
plt.show()

## 7. Importance des features (contribution aux clusters)

In [ ]:
# Contribution de chaque feature à la séparation des clusters
# = variance inter-clusters / variance totale
feature_importance = {}
for feat in FEATURES:
    total_var = df[feat].var()
    between_var = df.groupby('cluster')[feat].mean().var()
    feature_importance[feat] = between_var / total_var if total_var > 0 else 0

fi_series = pd.Series(feature_importance).sort_values(ascending=True)

plt.figure(figsize=(10, 6))
fi_series.plot(kind='barh', color='steelblue', edgecolor='white')
plt.title('Importance des features dans la séparation des clusters')
plt.xlabel('Variance inter-clusters / variance totale')
plt.tight_layout()
plt.show()

## 8. Résumé interprétatif

In [ ]:
summary = df.groupby('cluster').agg(
    nb_sessions=('id', 'count'),
    score_moyen=('score', 'mean'),
    reaction_moy_ms=('reaction_time_avg_ms', 'mean'),
    btn_press_rate_moy=('btn_press_rate', 'mean'),
    input_regularity_moy=('input_regularity', 'mean'),
    jeux=('game_id', lambda x: x.value_counts().index[0]),
    source_dominante=('source', lambda x: x.value_counts().index[0] if x.notna().any() else 'unknown'),
).round(2)

print('=== RÉSUMÉ DES CLUSTERS ===' )
print(summary.to_string())
print()
print('Interprétation suggérée :')
for c in sorted(df['cluster'].unique()):
    row = summary.loc[c]
    tags = []
    if row['score_moyen'] > summary['score_moyen'].median():
        tags.append('score élevé')
    else:
        tags.append('score faible')
    if row['btn_press_rate_moy'] > summary['btn_press_rate_moy'].median():
        tags.append('appuis fréquents')
    if row['reaction_moy_ms'] > 0 and row['reaction_moy_ms'] < summary[summary['reaction_moy_ms']>0]['reaction_moy_ms'].median():
        tags.append('réactif')
    if row['input_regularity_moy'] > summary['input_regularity_moy'].median():
        tags.append('régulier')
    print(f'  Cluster {c} ({int(row["nb_sessions"])} sessions) : {", ".join(tags)}')

In [ ]:
# Export avec labels clusters
df.to_csv('sessions_clustered.csv', index=False)
print('Fichier sessions_clustered.csv sauvegardé.')

In [ ]:
# --- PCA 2D par jeu — un subplot par game ---
from sklearn.decomposition import PCA

n_games = len(results)
fig, axes = plt.subplots(1, n_games, figsize=(6 * n_games, 6))
if n_games == 1:
    axes = [axes]

for ax, sub in zip(axes, results):
    game = sub['game_id'].iloc[0]

    # Re-normaliser pour PCA (même scaler que lors du clustering)
    scaler_v = StandardScaler()
    X_norm = scaler_v.fit_transform(sub[FEATURES_RETENUES].fillna(0))

    # PCA 2D
    n_comp = min(2, X_norm.shape[0] - 1, X_norm.shape[1])
    pca = PCA(n_components=n_comp, random_state=42)
    coords = pca.fit_transform(X_norm)
    var = pca.explained_variance_ratio_

    sub_plot = sub.copy()
    sub_plot['PC1'] = coords[:, 0]
    sub_plot['PC2'] = coords[:, 1] if n_comp > 1 else 0

    profils = sorted(sub_plot['profil'].unique())
    colors  = plt.cm.tab10(np.linspace(0, 1, len(profils)))
    palette = dict(zip(profils, colors))

    for profil, grp in sub_plot.groupby('profil'):
        ax.scatter(grp['PC1'], grp['PC2'],
                   label=profil, color=palette[profil],
                   s=130, alpha=0.85, edgecolors='white', linewidths=0.8)
        for _, row in grp.iterrows():
            ax.annotate(row['player_name'],
                        (row['PC1'], row['PC2']),
                        fontsize=7, alpha=0.7,
                        ha='center', va='bottom',
                        xytext=(0, 6), textcoords='offset points')

    sil = silhouette_score(X_norm, sub['cluster_id'])
    ax.set_xlabel(f'PC1 ({var[0]:.1%} var.)', fontsize=9)
    ax.set_ylabel(f'PC2 ({var[1]:.1%} var.)' if n_comp > 1 else 'PC2', fontsize=9)
    ax.set_title(f'{game.upper()}  |  silhouette={sil:.3f}', fontsize=11, pad=10)
    ax.legend(fontsize=8, framealpha=0.7)
    ax.grid(True, linestyle='--', alpha=0.4)

plt.suptitle('PCA 2D — clusters par jeu (couleur = profil, point = session)',
             fontsize=13, y=1.02)
plt.tight_layout()
plt.show()